# Deskriptive Analyse (inkl. min/max group sizes)

Wir laden die Ergebnisse und schauen zusätzlich, wie Distanz-Mittelwerte variieren mit `min_observations` und `max_observations`.


In [3]:
# Notebook-Block: Inhalt des geladenen Modells anzeigen

from joblib import load

# Pfad zum Modell
model_path = "C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/results/data/sex_models_best/eurasian_otter.joblib"

# Modell laden
clf = load(model_path)

# 1) Gesamtobjekt anzeigen
print("<< Pipeline Objekt >>")
print(clf)
print("\n")

# 2) Alle Schritte im Pipeline-Objekt
print("<< Pipeline Schritte >>")
for name, step in clf.named_steps.items():
    print(f"{name}: {step}")
print("\n")

# 3) Parameter des gesamten Pipeline-Objekts
print("<< Pipeline Parameter (get_params) >>")
params = clf.get_params(deep=True)
for key, value in params.items():
    print(f"{key}: {value}")


<< Pipeline Objekt >>
Pipeline(steps=[('transform', NumericTransformer()),
                ('outlier', OutlierCleanerTransformer(method='zscore')),
                ('select', FeatureSelectionTransformer(k=7, method='forward')),
                ('reduce_post',
                 DimensionalityReducerTransformer(method='umap',
                                                  n_components=10)),
                ('classifier',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsampl...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=None,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                 

In [6]:
import pandas as pd
from joblib import load

# Pfade definieren
model_path = "C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/results/data/sex_models_best/eurasian_otter.joblib"
data_path  = r"C:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\data\raw\Eurasian Otter.csv"
output_csv = r"C:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\results\data\Eurasian Otter_sex_pred.csv"

# Modell und Daten laden
clf = load(model_path)
df.columns = [
    col.replace('.', '_')
       .replace('-', '_')
       .replace('T', 't')
    for col in df.columns
]



# Direkt Vorhersagen aus der Pipeline holen
df["predicted_sex"]     = clf.predict(df)
probas = clf.predict_proba(df)
df["predicted_proba_f"] = probas[:, 0]
df["predicted_proba_m"] = probas[:, 1]

# Ergebnisse als CSV speichern
df.to_csv(output_csv, index=False)
print(f"Ergebnisse gespeichert in: {output_csv}")


c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Ergebnisse gespeichert in: C:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\results\data\Eurasian Otter_sex_pred.csv


In [ ]:
from FIT_python.caption_utils import save_caption
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 📥 Daten laden (Parquet!)
df = pd.read_parquet(
    r"C:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\results\data\pairwise_comp_all_species_experiment\amur_tiger_all_results_combined.parquet"
)

# 🔧 NA durch "none" ersetzen für methodische Spalten
df["outlier_method"] = df["outlier_method"].fillna("none")
df["scaler_method"] = df["scaler_method"].fillna("none")

# 🧠 Reduktion + Featureanzahl kombinieren
df["reducer_feat_group"] = df["reducer"].astype(str) + "_k" + df["feature_count"].astype(str)

# 🔍 Distanzmetriken extrahieren
dist_cols = [col for col in df.columns if col.startswith("dist_") and df[col].notna().all()]

# 🔁 Gruppierungsspalten für Multivariate Analyse
group_columns = [
    "selection_method", "reducer_feat_group", "outlier_method",
    "scaler_method", "use_sex", "n_components"
]

# 🧮 Funktion zur Berechnung der Trennungsgüte für jede Gruppenkombination
def multivariate_separation_score(data, group_cols):
    results = []
    grouped = data.groupby(group_cols)
    for group_vals, group_df in grouped:
        if group_df["same_individual"].nunique() < 2:
            continue  # beide Klassen notwendig
        true_mean = group_df[group_df.same_individual == True][dist_cols].mean()
        false_mean = group_df[group_df.same_individual == False][dist_cols].mean()
        delta = (false_mean - true_mean).rename("Δ")
        group_dict = dict(zip(group_cols, group_vals if isinstance(group_vals, tuple) else [group_vals]))
        result = pd.concat([pd.Series(group_dict), delta])
        results.append(result)
    return pd.DataFrame(results)

# 📊 Trennwerte berechnen
results_df = multivariate_separation_score(df, group_columns)

# 🧹 Nach Euclidean-Distanz sortieren (optional)
results_df = results_df.sort_values(by="dist_euclidean", ascending=False)

# 📈 Optional: Heatmap der Top-N Pipelines (nach mehreren Distanzen)
top_n = 20
heatmap_data = results_df.head(top_n).set_index([
    "selection_method", "reducer_feat_group", "outlier_method",
    "scaler_method", "use_sex", "n_components"
])[dist_cols]

# Heatmap anzeigen
plt.figure(figsize=(12, 8))
sns.heatmap(heatmap_data, annot=True, cmap="viridis", fmt=".2f", linewidths=0.5)
plt.title("Top-20 Pipeline-Kombinationen: Trennschärfe über alle Distanzen")
plt.ylabel("Pipeline (kombiniert)")
plt.xlabel("Distanzmetrik")
plt.tight_layout()
plt.show()

# 👉 Optional: Ergebnis als CSV exportieren
# results_df.to_csv("trennungsanalyse_pipeline_results.csv", index=False)


## 1 Mittlere Distanzen nach Gruppengrößen


In [5]:
from FIT_python.caption_utils import save_caption
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

# 📥 Daten laden
df = pd.read_parquet(
    r"C:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\results\data\pairwise_comp_all_species_experiment\amur_tiger_all_results_combined.parquet"
)

# 🔧 NAs als "none"
df["outlier_method"] = df["outlier_method"].fillna("none")
df["scaler_method"] = df["scaler_method"].fillna("none")

# 📌 Relevante Distanzmetriken
dist_cols = [col for col in df.columns if col.startswith("dist_") and df[col].notna().all()]

# 🔁 Gruppierung nur nach Pipeline
group_columns = ["pipeline"]

# 🧮 Funktion: berechnet Δ_mean und Δ_median
def multivariate_separation_score_with_median(data, group_cols):
    results = []
    grouped = data.groupby(group_cols)
    for group_vals, group_df in grouped:
        if group_df["same_individual"].nunique() < 2:
            continue

        true_dists = group_df[group_df.same_individual == True][dist_cols]
        false_dists = group_df[group_df.same_individual == False][dist_cols]

        mean_delta = (false_dists.mean() - true_dists.mean()).add_suffix("_mean")
        median_delta = (false_dists.median() - true_dists.median()).add_suffix("_median")

        group_dict = dict(zip(group_cols, group_vals if isinstance(group_vals, tuple) else [group_vals]))
        result = pd.concat([pd.Series(group_dict), mean_delta, median_delta])
        results.append(result)
    return pd.DataFrame(results)

# 📊 Berechnen
results_df = multivariate_separation_score_with_median(df, group_columns)

# 🪄 Skalierung (0–1) separat für Mean und Median
mean_cols = [col for col in results_df.columns if col.endswith("_mean")]
median_cols = [col for col in results_df.columns if col.endswith("_median")]

# Skalieren
scaler = MinMaxScaler()
scaled_mean = pd.DataFrame(scaler.fit_transform(results_df[mean_cols]), columns=[f"{col}_scaled" for col in mean_cols])
scaled_median = pd.DataFrame(scaler.fit_transform(results_df[median_cols]), columns=[f"{col}_scaled" for col in median_cols])

# Kombinieren
results_scaled = pd.concat([results_df["pipeline"].reset_index(drop=True), scaled_mean, scaled_median], axis=1)

# ➕ Mean/Medianscore berechnen
results_scaled["mean_score"] = scaled_mean.mean(axis=1)
results_scaled["median_score"] = scaled_median.mean(axis=1)

# 🔠 Sortieren nach Pipeline alphabetisch
results_sorted = results_scaled.sort_values(by="pipeline").reset_index(drop=True)

# 📈 Heatmap über alles
heatmap_data = results_sorted.set_index("pipeline")[
    list(scaled_mean.columns) + list(scaled_median.columns) + ["mean_score", "median_score"]
]

plt.figure(figsize=(16, max(10, len(heatmap_data) * 0.25)))
sns.heatmap(heatmap_data, annot=True, cmap="viridis", fmt=".2f", linewidths=0.3)
plt.title("Δ_mean & Δ_median pro Pipeline (skaliert, alphabetisch sortiert)")
plt.ylabel("Pipeline")
plt.xlabel("Distanzmetriken (Δ_mean / Δ_median, skaliert)")
plt.tight_layout()
plt.show()


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Frede\\OneDrive\\Projects and Disschaptors\\FIT_package\\results\\data\\pairwise_comp_all_species_experiment\\amur_tiger_all_results_combined.parquet'

In [ ]:
# Cell 3: Cross-Validation by Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, cross_val_score

# The three distance features you care about:
features = ["dist_euclidean", "dist_braycurtis", "dist_cosine"]

# Pull all unique pipelines from your results DataFrame
pipelines = df["pipeline"].unique()

# Use GroupKFold on comparison_id to prevent leakage across pair comparisons
gkf = GroupKFold(n_splits=5)
records = []

for pipe in pipelines:
    sub = df[df["pipeline"] == pipe]
    # skip small sets
    if len(sub) < 30:
        continue

    X = sub[features].values
    y = sub["same_individual"].astype(int).values
    groups = sub["comparison_id"].values





    clf = RandomForestClassifier(n_estimators=100, random_state=0)
    bal_scores = cross_val_score(
        clf, X, y,
        cv=gkf.split(X, y, groups=groups),
        scoring="balanced_accuracy",
        n_jobs=-1
    )
    auc_scores = cross_val_score(
        clf, X, y,
        cv=gkf.split(X, y, groups=groups),
        scoring="roc_auc",
        n_jobs=-1
    )

    records.append({
        "pipeline":        pipe,
        "bal_mean":        bal_scores.mean(),
        "bal_std":         bal_scores.std(),
        "auc_mean":        auc_scores.mean(),
        "auc_std":         auc_scores.std(),
        "n_pairs":         len(sub)
    })

res_df = pd.DataFrame(records)
res_df.sort_values("auc_mean", ascending=False).reset_index(drop=True)


In [ ]:
from FIT_python.caption_utils import save_caption
# %% 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

# –– 1) Assume you already have `res_df` loaded into memory
# If you need to read it from disk, uncomment the next line:
# res_df = pd.read_csv("path/to/your/all_results_combined.csv")

# –– 2) Make sure the flag column is boolean
res_df['use_sexmodel_prediction'] = res_df['use_sexmodel_prediction'].astype(bool)

# –– 3) Prepare the two groups
grouped = res_df.groupby('use_sexmodel_prediction')['bal_mean']
data_false = grouped.get_group(False)
data_true  = grouped.get_group(True)

# –– 4) Box-plot
fig, ax = plt.subplots(figsize=(6, 4))
ax.boxplot([data_false, data_true], labels=['sex_off', 'sex_on'])
ax.set_ylabel('Balanced accuracy (mean)')
ax.set_title('Effect of sex‐model features on Balanced Accuracy')
plt.tight_layout()
plt.show()

# %% 
# –– 5) Mann–Whitney U test
stat, pval = mannwhitneyu(data_false, data_true, alternative='two-sided')
print(f"Mann–Whitney U  stat={stat:.3f}, p-value={pval:.3e}")

if pval < 0.05:
    print("→ Significant difference in balanced accuracy (p<0.05)")
else:
    print("→ No significant difference detected (p≥0.05)")


In [ ]:
from FIT_python.caption_utils import save_caption
# %% Speichere Cluster-Labels und Dendrogramme für alle Distanzmaße
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import fcluster

# 1) CSV laden (Dein bereits eingelesenes df)
df = pd.read_csv(
    r"C:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\results\data\cluster_csv.csv",
    decimal=","
)

measures = [
    "dist_euclidean",
    "dist_manhattan",
    "dist_cosine",
    "dist_chebyshev",
    "dist_canberra",
    "dist_braycurtis",
    "dist_mahalanobis"
]

output_dir = Path("C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/results/data/clusters")
output_dir.mkdir(exist_ok=True)

for measure in measures:
    trails = sorted(set(df["trail_a_id"]) | set(df["trail_b_id"]))
    # Build complete distance matrix, fill NaNs with max
    D = pd.DataFrame(np.nan, index=trails, columns=trails)
    for a, b, val in zip(df["trail_a_id"], df["trail_b_id"], df[measure]):
        try:
            D.at[a, b] = D.at[b, a] = float(val)
        except:
            continue
    np.fill_diagonal(D.values, 0.0)
    max_d = np.nanmax(D.values)
    D_filled = D.fillna(max_d)

    # linkage
    condensed = squareform(D_filled.values)
    Z = hierarchy.linkage(condensed, method="average")

    # 2 Optionen zum Clustern:
    # a) nach fester Anzahl k
    k = 5
    labels_k = fcluster(Z, t=k, criterion='maxclust')
    # b) nach Distanz-Schwelle t
    t = max_d * 0.7
    labels_t = fcluster(Z, t=t, criterion='distance')

    # save labels
    df_labels = pd.DataFrame({
        "trail_id": D_filled.index,
        f"{measure}_cluster_k{k}": labels_k,
        f"{measure}_cluster_t{int(t*100)/100}": labels_t
    })
    df_labels.to_csv(output_dir / f"{measure}_clusters.csv", index=False)

    # save dendrogram
    plt.figure(figsize=(12,6))
    hierarchy.dendrogram(
        Z,
        labels=D_filled.index.tolist(),
        leaf_rotation=90,
        leaf_font_size=6,
        color_threshold=None
    )
    plt.title(f"Dendrogram {measure}")
    plt.ylabel("Distance")
    plt.tight_layout()
    plt.savefig(output_dir / f"{measure}_dendrogram.png", dpi=150)
caption = 'Plot generated'
save_caption(output_dir / f"{measure}_dendrogram.png", caption)
    plt.close()


In [ ]:
!pip install networkx 

In [ ]:
from FIT_python.caption_utils import save_caption
# %% Konstellationsdiagramm per MDS + kNN-Netzwerk
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

from sklearn.manifold import MDS
from sklearn.neighbors import NearestNeighbors

# 1) CSV mit Distanzen laden
df = pd.read_csv(
    r"C:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\results\data\cluster_csv.csv",
    decimal=","
)

measure = "dist_euclidean"
# Sammle alle Trail-IDs
trails = sorted(set(df["trail_a_id"]) | set(df["trail_b_id"]))
n = len(trails)

# 2) Distanzmatrix aufbauen und fehlende mit max füllen
D = pd.DataFrame(np.nan, index=trails, columns=trails)
for a, b, val in zip(df["trail_a_id"], df["trail_b_id"], df[measure]):
    D.at[a, b] = D.at[b, a] = float(val)
np.fill_diagonal(D.values, 0.0)
maxd = np.nanmax(D.values)
D = D.fillna(maxd)

# 3) MDS-Projektion in 2D
mds = MDS(n_components=2, dissimilarity="precomputed", random_state=42)
coords = mds.fit_transform(D.values)

# 4) k-Nearest-Neighbor‐Graph (z.B. k=3)
k = 3
nn = NearestNeighbors(n_neighbors=k+1, metric="precomputed")
nn.fit(D.values)
# Für jeden Punkt die Indizes der nächsten Nachbarn (inkl. sich selbst an Stelle 0)
distances, indices = nn.kneighbors(D.values)

G = nx.Graph()
# Knoten mit Position
for i, label in enumerate(trails):
    G.add_node(label, pos=(coords[i,0], coords[i,1]))

# Kanten zu den k nächsten Nachbarn
for i, nbrs in enumerate(indices):
    src = trails[i]
    for j in nbrs[1:]:  # skip den ersten Eintrag (sich selbst)
        tgt = trails[j]
        G.add_edge(src, tgt, weight=D.at[src, tgt])

# 5) Plot
pos = nx.get_node_attributes(G, 'pos')
plt.figure(figsize=(8,6))
# Kanten nach Gewicht einfärben / dick zeichnen
weights = [G[u][v]['weight'] for u,v in G.edges()]
# normalisieren für Breite
maxw = max(weights)
widths = [2*(1 - w/maxw) + 0.5 for w in weights]

nx.draw_networkx_nodes(G, pos, node_size=50, node_color='steelblue')
nx.draw_networkx_edges(G, pos, width=widths, edge_color='gray', alpha=0.7)
# optional Labels
# nx.draw_networkx_labels(G, pos, font_size=6)

plt.title(f"Konstellationsdiagramm (MDS + {k}-NN) mit {measure}")
plt.axis('off')
plt.tight_layout()
plt.savefig(output_dir / f"{measure}_konstellation.png", dpi=150)
caption = 'Plot generated'
save_caption(output_dir / f"{measure}_konstellation.png", caption)

plt.show()


In [ ]:
from FIT_python.caption_utils import save_caption
# Cell 4: Confusion matrices for top-5 pipelines
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

# pick top 5 by AUC
top5 = res_df.nlargest(5, "auc_mean")["pipeline"].tolist()

gkf = GroupKFold(n_splits=5)

for pipe in top5:
    sub = df[df["pipeline"] == pipe]
    X = sub[features].values
    y = sub["same_individual"].astype(int).values
    groups = sub["comparison_id"].values

    clf = RandomForestClassifier(n_estimators=100, random_state=0)
    # get cross‐val predictions, grouping by comparison_id
    y_pred = cross_val_predict(
        clf, X, y,
        cv=gkf.split(X, y, groups),
        n_jobs=-1
    )

    cm = confusion_matrix(y, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["diff_ind","same_ind"])
    
    fig, ax = plt.subplots(figsize=(4,4))
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"{pipe}\nAUC={res_df.set_index('pipeline').loc[pipe,'auc_mean']:.2f}")
    plt.tight_layout()
    plt.show()


In [ ]:
from FIT_python.caption_utils import save_caption
# Cell X: Balanced accuracy vs. min/max observations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import balanced_accuracy_score

# assume df_full (the full results DataFrame) and res_df (the grid‐search summary) are in memory,
# and that df_full has columns: 
#   ['pipeline','comparison_id','same_individual','min_observations','max_observations','dist_braycurtis','dist_cosine',...]
# and that you have already identified your top-5 pipelines:
top5 = res_df.nlargest(5, "auc_mean")["pipeline"].tolist()

for pipe in top5:
    sub = df[df["pipeline"] == pipe].copy()
    # build the feature matrix for the classifier
    X = sub[["dist_braycurtis","dist_cosine"]].values
    y = sub["same_individual"].astype(int).values
    groups = sub["comparison_id"].values

    # out-of-fold predictions
    gkf = GroupKFold(n_splits=5)
    clf = RandomForestClassifier(n_estimators=100, random_state=0)
    y_pred = cross_val_predict(clf, X, y,
                               cv=gkf.split(X, y, groups),
                               n_jobs=-1)

    sub["y_pred"] = y_pred

    # compute balanced accuracy per min_observations and max_observations
    ba_min = []
    for m in sorted(sub["min_observations"].unique()):
        grp = sub[sub["min_observations"] == m]
        if len(grp) < 10: 
            continue
        ba_min.append((m,
                       balanced_accuracy_score(grp["same_individual"], grp["y_pred"])))
    ba_min = pd.DataFrame(ba_min, columns=["group_size","balanced_accuracy"])

    ba_max = []
    for M in sorted(sub["max_observations"].unique()):
        grp = sub[sub["max_observations"] == M]
        if len(grp) < 10:
            continue
        ba_max.append((M,
                       balanced_accuracy_score(grp["same_individual"], grp["y_pred"])))
    ba_max = pd.DataFrame(ba_max, columns=["group_size","balanced_accuracy"])

    # plot
    fig, ax = plt.subplots(figsize=(7,4))
    ax.scatter(ba_min["group_size"], ba_min["balanced_accuracy"],
               label="min_observations", s=50)
    ax.scatter(ba_max["group_size"], ba_max["balanced_accuracy"],
               marker="x", label="max_observations", s=60)
    ax.set_xlabel("Group size")
    ax.set_ylabel("Balanced accuracy")
    ax.set_title(f"{pipe}\nBalanced accuracy vs. min/max observations")
    ax.set_ylim(0.0,1.02)
    ax.legend()
    plt.tight_layout()
    plt.show()

